In [ ]:
# Muons in the CORSIKA showers (make tables);  writes nmuons.yaml, read by model.py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml

# epsilon straight from model.yaml:  model.py needs the nmuons.yaml written here
epsilon = float(yaml.safe_load(open("model.yaml"))["epsilon"])
ev = pd.read_feather("../showers/tables/finecore.feather")
ev["E_eV"] = ev.E_GeV * 1e9
print(f"{len(ev)} showers at {ev.E_eV.nunique()} energies, {ev.E_eV.min():.2g}-{ev.E_eV.max():.2g} eV")

In [ ]:
# Muon number per energy (median and 16-84% band) and its power-law fit
g = ev.groupby("E_eV").n_muons
E, med, p16, p84 = g.median().index.values, g.median().values, g.quantile(0.16).values, g.quantile(0.84).values
eta, lnbeta = np.polyfit(np.log(E / epsilon), np.log(med), 1)
beta = np.exp(lnbeta)
print(f"N(E) = beta (E / epsilon)^eta:  beta {beta:.4e}  eta {eta:.4f}  (epsilon {epsilon:g} eV)")

In [ ]:
# Write nmuons.yaml:  the fit, and the points for the comparison plot in full_analytic
def flow(x):
    """Signed exponents, so YAML reads floats."""
    return "[" + ", ".join(f"{v:.4e}" for v in x) + "]"

with open("nmuons.yaml", "w") as f:
    f.write("# N(E) = beta (E / epsilon)^eta, fit to the median n_muons of the CORSIKA showers (aaa_muons.ipynb)\n"
            f"beta: {beta:.4e}\neta: {eta:.4f}\n"
            "# n_muons per shower at each CORSIKA energy:  median and the 16-84 percentile band\n"
            f"E: {flow(E)}      # eV\nmedian: {flow(med)}\np16: {flow(p16)}\np84: {flow(p84)}\n")
print(f"wrote nmuons.yaml:  {len(E)} energies")

In [ ]:
# Muon number:  all showers, median and band, the fit, and the ratio to the fit
fit = beta * (E / epsilon)**eta
fig, axes = plt.subplots(2, 1, figsize=(7, 6.5), sharex=True, gridspec_kw={"height_ratios": [3, 1]})
axes[0].plot(ev.E_eV, ev.n_muons, ",", color="0.6", label="showers")
axes[0].fill_between(E, p16, p84, color="C0", alpha=0.3, linewidth=0, label="16-84%")
axes[0].plot(E, med, ".", color="C0", markersize=4, label="median")
axes[0].plot(E, fit, "k-", label=f"fit:  {beta:.3g} (E / {epsilon:.0e} eV)$^{{{eta:.4f}}}$")
axes[0].set_yscale("log"); axes[0].set_ylabel("muons per shower")
axes[0].set_title("Muon number"); axes[0].legend()
axes[1].fill_between(E, p16 / fit, p84 / fit, color="C0", alpha=0.3, linewidth=0)
axes[1].plot(E, med / fit, ".", color="C0", markersize=4)
axes[1].axhline(1, color="k", linewidth=0.8)
axes[1].set_xscale("log"); axes[1].set_xlabel("E (eV)"); axes[1].set_ylabel("ratio to fit")
plt.tight_layout(); plt.savefig("pdf/muon_number.pdf"); plt.show()